# Find closest sequence

Typically, ClinVar uses only one or a few reference sequences for a human gene.  We
should find the sequences from other species that are most similar to the human
reference sequence.  

To illustrate, let's consider ABCD1.  Below we see that there are 2,067 variants
in the ClinVar database. 

**Note:** I had to update the code for the `get_variant_details()` function.
The `@RecordType` appears to have changed to `classified`.  I suspect that this
is due to the expansion of the database to include both germline and somatic
mutations.  The new function needs more comprehensive testing but it seems to
work fine on the ABCD1 variants.

In [3]:
run kondrashov

In [4]:
gene = "HBB"

In [35]:
nvars, varids = get_variant_ids(gene)
nvars
print(len(varids), varids)
#varids['15090']
type(varids)

1962 ['4853115', '4853074', '4851147', '4849097', '4848400', '4848053', '4848014', '4847669', '4819685', '4818727', '4818706', '4818705', '4818704', '4818703', '4818702', '4798543', '4795169', '4771045', '4766669', '4753219', '4735712', '4734723', '4734636', '4729396', '4695393', '4692970', '4689563', '4687131', '4685642', '4539279', '4537333', '4536042', '4536039', '4535916', '4532971', '4532970', '4532969', '4532968', '4531951', '4525835', '4280318', '4280311', '4280310', '4279099', '4277917', '4078907', '4075915', '4075885', '4074264', '4074263', '4071470', '4069120', '4069118', '4069117', '4069116', '4069115', '4069114', '4069112', '4069111', '4069110', '4069109', '4069108', '4069107', '4069106', '4069105', '4069104', '3911998', '3910010', '3910009', '3910008', '3907482', '3902791', '3896213', '3896065', '3781330', '3781328', '3777010', '3769508', '3769274', '3769063', '3769040', '3768611', '3767085', '3767038', '3766927', '3766686', '3766488', '3766487', '3701912', '3693356', '367

Bio.Entrez.Parser.ListElement

In [ ]:
## confirm that output is similar to variants in 'pathogenic/'HBB.csv''
x= '4277917'
if x in varids:
    print(x)
else:
    print(x, 'not found')

4277917


In [34]:
from tqdm import tqdm

In [ ]:
###### This checks each Clinvar variant per gene in 'pathogenic' directory and identifies transcripts other than NM_000033.4 that encode each variant
### Then I sift througth to retain unique transcripts

import pandas as pd
ABCD1_df= pd.read_csv("pathogenic/ABCD1.csv")
varids = ABCD1_df["VariationID"].astype(str).tolist()
for varid in tqdm(varids):
    varrec = get_variant(varid)
    name, acc, mut, pmut, muttype, status = get_variant_details(varrec)
    if acc != "NM_000033.4":
        #print(varid, acc, name, mut, pmut, muttype, status)



### Step 1 identify reference human sequence
##### I check each Clinvar variant per gene in 'pathogenic' directory and sift througth to identify unique transcripts

In [96]:
import pandas as pd
from tqdm import tqdm

ABCD1_df = pd.read_csv("pathogenic/ABCD1.csv")
varids = ABCD1_df["VariationID"].astype(str).tolist()

unique_acc = set()

for varid in tqdm(varids):
    varrec = get_variant(varid)
    name, acc, mut, pmut, muttype, status = get_variant_details(varrec)

    if acc is not None:
        unique_acc.add(acc)

print(f"Found {len(unique_acc)} unique accession(s):")
print(sorted(unique_acc))

100%|██████████| 327/327 [02:27<00:00,  2.21it/s]

Found 1 unique accession(s):
['NM_000033.4']


### STEP 2 Obtain corresponding protein of each unique human transcript

In [ ]:
run kondrashov

In [137]:
transcript_record = get_transcript("NM_000033.4")
xx= get_protein_from_transcript(transcript_record)
xx

(Seq('atgccggtgctctccaggccccggccctggcgggggaacacgctgaagcgcacg...tga'),
 Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
 'NP_000024.2',
 411,
 2649)

# Human sequences

There are 3 unique human ABCD1 protein sequences in the `fasta` file.

In [138]:
hum_ids = ['NP_000024.2']

In [141]:
# Collect unique human sequences
print('{0} human sequences\n'.format(gene))
hum_sequences = []
hum_records = []
inc = 0
exc = 0
with Entrez.efetch(
    db="protein", rettype="gb", retmode="text", id=hum_ids,
) as handle:
    for seq_record in SeqIO.parse(handle, "gb"):
        sp = get_species(seq_record)
        seq = seq_record.seq
        if seq in hum_sequences:
            print("\t{0}\t({1} aa)\t{2}\t*** the same as sequence {3} (excluded) ***".format(seq_record.id, len(seq), seq_record.description, hum_sequences.index(seq)))
            exc += 1
        else:
            hum_sequences.append(seq)
            print("{3}:\t{0}\t({1} aa)\t{2}".format(seq_record.id, len(seq), seq_record.description, inc))
            hum_records.append(seq_record)
            inc += 1

HBB human sequences

0:	NP_000024.2	(745 aa)	ATP-binding cassette sub-family D member 1 isoform 1 [Homo sapiens]


In [142]:
hum_sequences[0]

Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST')

### Step 3 Read through each fasta file in 'fasta/' folder
#### Collect unique human sequences

In [161]:
gene= 'ABCD1'
print('{0} orthologs: all species sequences\n'.format(gene))
all_sequences = []
seq_records = []
inc = 0
exc = 0
for seq_record in SeqIO.parse('fasta/ABCD1.fasta', "fasta"):
    sp = get_species(seq_record)
    seq = seq_record.seq
    if seq in all_sequences:
        print("\t{0}\t({1} aa)\t{2}\t*** the same as sequence {3} (excluded) ***".format(seq_record.id, len(seq), seq_record.description, all_sequences.index(seq)))
        exc += 1
    else:
        all_sequences.append(seq)
        print(seq_record.id, len(seq), seq_record.description, inc, sep= '\t')
        seq_records.append(seq_record)
        inc += 1


ABCD1 orthologs: all species sequences

NP_000024.2	745	NP_000024.2 ATP-binding cassette sub-family D member 1 isoform 1 [Homo sapiens]	0
NP_001427676.1	845	NP_001427676.1 ATP-binding cassette sub-family D member 1 isoform 2 [Homo sapiens]	1
XP_047297873.1	575	XP_047297873.1 ATP-binding cassette sub-family D member 1 isoform X2 [Homo sapiens]	2
NP_002981.2	93	NP_002981.2 C-C motif chemokine 22 precursor [Homo sapiens]	3
XP_047290405.1	106	XP_047290405.1 C-C motif chemokine 22 isoform X1 [Homo sapiens]	4
XP_054169574.1	106	XP_054169574.1 C-C motif chemokine 22 isoform X1 [Homo sapiens]	5
XP_054169575.1	93	XP_054169575.1 C-C motif chemokine 22 isoform X2 [Homo sapiens]	6
XP_001085640.2	745	XP_001085640.2 ATP-binding cassette sub-family D member 1 isoform X2 [Macaca mulatta]	7
XP_028698141.1	734	XP_028698141.1 ATP-binding cassette sub-family D member 1 isoform X3 [Macaca mulatta]	8
XP_077844975.1	757	XP_077844975.1 ATP-binding cassette sub-family D member 1 isoform X1 [Macaca mulatta]	9
X

In [159]:
all_sequences

[Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
 'NP_000024.2',
 Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
 'NP_001427676.1',
 Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...VTR'),
 'XP_047297873.1',
 Seq('MDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'),
 'NP_002981.2',
 Seq('MDMTPGLRHTGQSMDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'),
 'XP_047290405.1',
 Seq('MDMTPGLRHTGQSMARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'),
 'XP_054169574.1',
 Seq('MARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'),
 'XP_054169575.1',
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AST'),
 'XP_001085640.2',
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...WPM'),
 'XP_028698141.1',
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AGL'),
 'XP_077844975.1',
 Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
 'XP_016800023.1',
 Seq('MPVLSRPRPWRGSTLKRTAVL

### STEP 3   Finding the most similar *species* sequence to the reference human sequence

The following defines a pairwise aligner object using the Needleman-Wunsch
algorithm using the default `blastp` scoring scheme and substitution matrix.

In [146]:
from Bio import SeqIO, Align
from Bio.Seq import Seq

In [147]:
aligner = Align.PairwiseAligner(scoring="blastp")
print(aligner)

Pairwise sequence aligner with parameters
  substitution_matrix: <Array object at 0x1a7e76873f0>
  open_internal_insertion_score: -12.000000
  extend_internal_insertion_score: -1.000000
  open_left_insertion_score: -12.000000
  extend_left_insertion_score: -1.000000
  open_right_insertion_score: -12.000000
  extend_right_insertion_score: -1.000000
  open_internal_deletion_score: -12.000000
  extend_internal_deletion_score: -1.000000
  open_left_deletion_score: -12.000000
  extend_left_deletion_score: -1.000000
  open_right_deletion_score: -12.000000
  extend_right_deletion_score: -1.000000
  mode: global



In [148]:
print(aligner.substitution_matrix[:, :])

     A    B    C    D    E    F    G    H    I    J    K    L    M    N    O    P    Q    R    S    T    U    V    W    X    Y    Z    *
A  4.0 -2.0  0.0 -2.0 -1.0 -2.0  0.0 -2.0 -1.0 -1.0 -1.0 -1.0 -1.0 -2.0 -1.0 -1.0 -1.0 -1.0  1.0  0.0  0.0  0.0 -3.0 -1.0 -2.0 -1.0 -4.0
B -2.0  4.0 -3.0  4.0  1.0 -3.0 -1.0  0.0 -3.0 -3.0  0.0 -4.0 -3.0  4.0 -1.0 -2.0  0.0 -1.0  0.0 -1.0 -3.0 -3.0 -4.0 -1.0 -3.0  0.0 -4.0
C  0.0 -3.0  9.0 -3.0 -4.0 -2.0 -3.0 -3.0 -1.0 -1.0 -3.0 -1.0 -1.0 -3.0 -1.0 -3.0 -3.0 -3.0 -1.0 -1.0  9.0 -1.0 -2.0 -1.0 -2.0 -3.0 -4.0
D -2.0  4.0 -3.0  6.0  2.0 -3.0 -1.0 -1.0 -3.0 -3.0 -1.0 -4.0 -3.0  1.0 -1.0 -1.0  0.0 -2.0  0.0 -1.0 -3.0 -3.0 -4.0 -1.0 -3.0  1.0 -4.0
E -1.0  1.0 -4.0  2.0  5.0 -3.0 -2.0  0.0 -3.0 -3.0  1.0 -3.0 -2.0  0.0 -1.0 -1.0  2.0  0.0  0.0 -1.0 -4.0 -2.0 -3.0 -1.0 -2.0  4.0 -4.0
F -2.0 -3.0 -2.0 -3.0 -3.0  6.0 -3.0 -1.0  0.0  0.0 -3.0  0.0  0.0 -3.0 -1.0 -4.0 -3.0 -3.0 -2.0 -2.0 -2.0 -1.0  1.0 -1.0  3.0 -3.0 -4.0
G  0.0 -1.0 -3.0 -1.0 -2.0 -3.0  6.0 -2.0

The following shows that the second sequence is the most similar to the human
sequence.  It makes sense because it is the only one that has the same length.

In [169]:
hum_sequences
all_sequences

[Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
 Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
 Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...VTR'),
 Seq('MDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'),
 Seq('MDMTPGLRHTGQSMDRLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'),
 Seq('MDMTPGLRHTGQSMARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYV...LSQ'),
 Seq('MARLQTALLVVLVLLAVALQATEAGPYGANMEDSVCCRDYVRYRLPLRVVKHFY...LSQ'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AST'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...WPM'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPARGSQAPAREPT...AGL'),
 Seq('MPVLSRPRPWRGNTLKRTAVLLALAAYGAHKVYPLVRQCLAPARGLQAPAGEPT...AST'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AST'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...AGL'),
 Seq('MPVLSRPRPWRGSTLKRTAVLLALAAYGVHKVYPLVRQCLAPVRGSQAPAREPT...WPM'),
 Seq('MPVLSRPRPWRGST

In [162]:
max_score = aligner.score(hum_sequences[0], hum_sequences[0])
for i in range(5):
    score = aligner.score(hum_sequences[0], all_sequences[i])
    score

In [175]:
max_score = aligner.score(hum_sequences[0], hum_sequences[0])
for i in range(50):
    score = aligner.score(hum_sequences[0], all_sequences[i])
    print(f"{seq_ids}: {score/max_score:.4f}")

XP_063577189.1: 1.0000
XP_063577189.1: 0.9709
XP_063577189.1: 0.5691
XP_063577189.1: -0.1600
XP_063577189.1: -0.1563
XP_063577189.1: -0.1560
XP_063577189.1: -0.1600
XP_063577189.1: 0.9806
XP_063577189.1: 0.9549
XP_063577189.1: 0.9525
XP_063577189.1: 0.9976
XP_063577189.1: 0.9795
XP_063577189.1: 0.9515
XP_063577189.1: 0.9538
XP_063577189.1: 0.6022
XP_063577189.1: 0.5793
XP_063577189.1: 0.9418
XP_063577189.1: 0.9114
XP_063577189.1: 0.9266
XP_063577189.1: 0.9158
XP_063577189.1: 0.9853
XP_063577189.1: 0.8193
XP_063577189.1: 0.4684
XP_063577189.1: 0.9874
XP_063577189.1: 0.9837
XP_063577189.1: 0.8891
XP_063577189.1: 0.9027
XP_063577189.1: 0.9137
XP_063577189.1: 0.9177
XP_063577189.1: 0.9321
XP_063577189.1: 0.9848
XP_063577189.1: 0.4694
XP_063577189.1: 0.9557
XP_063577189.1: 0.9822
XP_063577189.1: 0.9614
XP_063577189.1: 0.9801
XP_063577189.1: 0.9544
XP_063577189.1: 0.9588
XP_063577189.1: 0.9263
XP_063577189.1: 0.8429
XP_063577189.1: 0.9308
XP_063577189.1: 0.9195
XP_063577189.1: 0.9583
XP_0635